In [30]:
import torch
import numpy as np
import pandas as pd
import sys

# from visualise import plot_area

In [15]:
# setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cuda



# Combined tensor

- [0, :, : ] bed elevation
- [1, :, : ] surface elevation
- [2, :, : ] thickess elevation
- [3, :, : ] mask
- [4, :, : ] VY
- [5, :, : ] VX
- [6, :, : ] Y (indexed high to low)
- [7, :, : ] X (low to high)
- [8, :, : ] speed
- [9, :, : ] angle

In [16]:
# import tensor from local
bm_vel = torch.load('/home/kim/data/bedmachine_phase_velocity_tensor.pt').to(device)

In [17]:
def subset_bedmachine(y_min, y_max, x_min, x_max, bedmachine_tensor):
    # Inclusive of min and max
    # Must be on grid
    # Get indices
    print("Km in y direction:", (y_max - y_min)/1000)
    print("Km in x direction:", (x_max - x_min)/1000)
    # y
    y_min_index = (bedmachine_tensor[6, :, 0] == y_min).nonzero().item()
    y_max_index = (bedmachine_tensor[6, :, 0] == y_max).nonzero().item()
    # x
    x_min_index = (bedmachine_tensor[7, 0, :] == x_min).nonzero().item()
    x_max_index = (bedmachine_tensor[7, 0, :] == x_max).nonzero().item()
    # Y has to be from max index to min index
    bedmachine_tensor_subset = bedmachine_tensor[:, y_max_index : (y_min_index + 1), x_min_index : (x_max_index + 1)]

    # Calculate speed
    bedmachine_tensor_subset = torch.cat((bedmachine_tensor_subset, 
                                          torch.sqrt(bedmachine_tensor_subset[4, :, :]**2 + bedmachine_tensor_subset[5, :,:]**2).unsqueeze(0),
                                          torch.arctan(bedmachine_tensor_subset[4, :, :]/bedmachine_tensor_subset[5, :,:]).unsqueeze(0)
                                          ), dim = 0).to(device)

    print("Correlation coefficients:")
    pd.set_option('display.precision', 2)
    display(pd.DataFrame(data = torch.corrcoef(
        bedmachine_tensor_subset.reshape(10, -1)[[0, 1, 2, 3, 4, 5, 8, 9]]).cpu().numpy(),
                       index = ["bed", "surface", "thickness", "mask", "vy", "vx", "speed", "angle"],
                       columns = ["bed", "surface", "thickness", "mask", "vy", "vx", "speed", "angle"]))

    # plot_area(y_min, y_max, x_min, x_max)

    return bedmachine_tensor_subset

In [18]:
out = subset_bedmachine(y_min = -1000000, y_max = 1000000, x_min = 000000, x_max = 2000000, bedmachine_tensor = bm_vel)

Km in y direction: 2000.0
Km in x direction: 2000.0
Correlation coefficients:


,bed,surface,thickness,mask,vy,vx,speed,angle
bed,1.00,0.33,-0.41,-0.31,NaN,NaN,NaN,NaN
surface,0.33,1.00,0.72,-0.40,NaN,NaN,NaN,NaN
thickness,-0.41,0.72,1.00,-0.21,NaN,NaN,NaN,NaN
mask,-0.31,-0.40,-0.21,1.00,NaN,NaN,NaN,NaN
vy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vx,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
speed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
angle,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
"""
import matplotlib.pyplot as plt  # plotting library
import matplotlib.patches as mpatches # draw domain boundry patches

# Issue: Maybe downgrade shapely pip install shapely==1.8.5
import cartopy.crs as ccrs  # Projections list
import cartopy.feature as cfeature # for coastlines


def plot_area(y_min, y_max, x_min, x_max):
    # Plot area in Antarctica w.r.t. on continental-scale map. 
    #
    # Args:
    #    y_min (_type_): Polar stereographic y min
    #    y_max (_type_): Polar stereographic
    #    x_min (_type_): Polar stereographic
    #    x_max (_type_): Polar stereographic
    
    special_color = '#2D27EB'
    
    # Initialise plot
    fig = plt.figure(figsize = [6, 6])
    ax = plt.axes(projection = ccrs.SouthPolarStereo())

    # restrict to over 65 lat
    ax.set_extent([-180, 180, -90, -65], ccrs.PlateCarree())

    # hides boundry line
    ax.axis('off')

    # add grey land
    ax.add_feature(cfeature.LAND, facecolor = ("#FAFAFA"), alpha = 1.0)

    # thin coastline lines
    ax.add_feature(cfeature.COASTLINE, edgecolor = special_color, linestyle = '-', linewidth = 0.4, alpha = 0.7)

    # patch
    ax.add_patch(mpatches.Rectangle(xy = [x_min, y_min], width = (x_max - x_min), height = (y_max - y_min),
                                facecolor = 'none', edgecolor = special_color, linewidth = 0.8,
                                transform = ccrs.SouthPolarStereo()))
    plt.show()

"""

'\nimport matplotlib.pyplot as plt  # plotting library\nimport matplotlib.patches as mpatches # draw domain boundry patches\n\n# Issue: Maybe downgrade shapely pip install shapely==1.8.5\nimport cartopy.crs as ccrs  # Projections list\nimport cartopy.feature as cfeature # for coastlines\n\n\ndef plot_area(y_min, y_max, x_min, x_max):\n    # Plot area in Antarctica w.r.t. on continental-scale map. \n    #\n    # Args:\n    #    y_min (_type_): Polar stereographic y min\n    #    y_max (_type_): Polar stereographic\n    #    x_min (_type_): Polar stereographic\n    #    x_max (_type_): Polar stereographic\n    \n    special_color = \'#2D27EB\'\n    \n    # Initialise plot\n    fig = plt.figure(figsize = [6, 6])\n    ax = plt.axes(projection = ccrs.SouthPolarStereo())\n\n    # restrict to over 65 lat\n    ax.set_extent([-180, 180, -90, -65], ccrs.PlateCarree())\n\n    # hides boundry line\n    ax.axis(\'off\')\n\n    # add grey land\n    ax.add_feature(cfeature.LAND, facecolor = ("#FAFAFA

In [20]:
# Missing values per variable
# As percentage
torch.sum(out.isnan(), dim = (1, 2))/out.reshape(10, -1).shape[-1]

tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0004, 0.0004, 0.0000, 0.0000, 0.0004,
        0.0004], device='cuda:0')

In [21]:
wave_y_min = - 601500
wave_x_min = 1034000
wave_y_max = -572000
wave_x_max = 1063500 # 29500 midpoint distance

wave = subset_bedmachine(y_min = wave_y_min, y_max = wave_y_max, x_min = wave_x_min, x_max = wave_x_max, bedmachine_tensor = bm_vel)

# bed
wave_bed_tri = calculate_TRI(wave[0, :, :].unsqueeze(0))
wave_surface_tri = calculate_TRI(wave[1, :, :].unsqueeze(0))
print("Mean bed 5x5 TRI:", np.round(torch.mean(wave_bed_tri).item(), 2))
print("Mean surface 5x5 TRI:", np.round(torch.mean(wave_surface_tri).item(), 2))

Km in y direction: 29.5
Km in x direction: 29.5
Correlation coefficients:


,bed,surface,thickness,mask,vy,vx,speed,angle
bed,1.00,0.04,-0.49,NaN,0.15,-0.49,0.48,-0.49
surface,0.04,1.00,0.85,NaN,0.37,0.67,-0.74,0.59
thickness,-0.49,0.85,1.00,NaN,0.24,0.84,-0.89,0.77
mask,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vy,0.15,0.37,0.24,NaN,1.00,-0.18,0.03,-0.34
vx,-0.49,0.67,0.84,NaN,-0.18,1.00,-0.99,0.98
speed,0.48,-0.74,-0.89,NaN,0.03,-0.99,1.00,-0.95
angle,-0.49,0.59,0.77,NaN,-0.34,0.98,-0.95,1.00


Mean bed 5x5 TRI: 7.51
Mean surface 5x5 TRI: 8.31


- Mask covar is NaN because it's variance is 0 in this case. (All are 2.)

In [23]:
def calculate_TRI(input_tensor, kernel_size = torch.tensor(5)):
    """Returns tensor of same size with tri values: Values on boarders are naturally small sinze it is not the full window size. Default is a 5x5 TRI (Terrain Ruggedness Index).

    Args:
        input_tensor (_type_, optional): [1, N, N]
        kernel_size (int, optional): _description_. Defaults to 5.

    Returns:
        tri_tensor
    """
    tri_tensor = torch.zeros(size = input_tensor.shape).to(device)
    half_size = ((kernel_size - 1.0)/2.0).int()

    # Rows
    for r in range(tri_tensor.shape[-2]):
        # Columns
        for c in range(tri_tensor.shape[-1]):
            r_min = np.max([0, r - half_size])
            r_max = np.min([(tri_tensor.shape[-2] - 1) , r + half_size]) # -1 for index versus length
            c_min = np.max([0, c - half_size])
            c_max = np.min([(tri_tensor.shape[-1] - 1), c + half_size])
            # window tensor
            window = input_tensor[0, r_min:(r_max + 1), c_min:(c_max + 1)].to(device) # +1 to include c_max and r_max
            reference_pixel = input_tensor[0, r, c].to(device)
            tri = torch.sub(input = window, other = reference_pixel).to(device)
            tri = torch.pow(input = tri, exponent = 2).to(device)
            tri = torch.sum(tri).to(device)
            tri = torch.sqrt(tri).to(device)
            # Assign 
            tri_tensor[0, r, c] = tri

    return tri_tensor


test_tensor = bm_vel[0, 0:10, 0:10].unsqueeze(0)
calculate_TRI(test_tensor)

tensor([[[230.8425, 346.5834, 252.9445, 249.6828, 246.1077, 260.8087, 224.5011,
          260.7348, 250.9322, 219.2190],
         [208.9709, 229.7915, 259.2351, 280.2643, 274.5108, 257.7036, 265.6740,
          295.4511, 248.0283, 217.4245],
         [271.3902, 291.2957, 321.2961, 318.1880, 324.0915, 332.8163, 347.7728,
          382.0014, 338.2438, 340.0506],
         [222.6096, 223.8598, 266.4333, 278.8742, 313.0096, 335.1180, 361.1006,
          406.3945, 352.1302, 329.3397],
         [210.7903, 194.4436, 236.5233, 247.3039, 300.4958, 346.2194, 397.8142,
          469.2066, 419.2865, 381.8120],
         [153.3463, 173.7046, 206.4382, 223.7266, 280.1905, 343.9196, 413.3529,
          495.7029, 446.0604, 419.3822],
         [144.4314, 167.8350, 188.1465, 215.9804, 284.7379, 333.6214, 405.6846,
          486.0594, 466.3382, 514.9872],
         [124.1475, 161.1246, 177.9510, 205.9578, 282.4376, 347.6404, 378.7322,
          441.0911, 415.5928, 458.6687],
         [ 83.9051, 127.0269, 16

# Dome A

In [25]:
scene_size = 30000 # in meters
n_scenes = 20 # scenes per row/column: 400 scences per domain
span = scene_size * n_scenes

# stick to order: y, x
domea_y_min = 0
domea_x_min = 899000
domea_y_max = 600000
domea_x_max = 1499000

domea = subset_bedmachine(y_min = domea_y_min, 
                          y_max = domea_y_max, 
                          x_min = domea_x_min, 
                          x_max = domea_x_max, 
                          bedmachine_tensor = bm_vel)

# bed
# takes 13 min on GPU
domea_bed_tri = calculate_TRI(domea[0, :, :].unsqueeze(0))
print("Done")
domea_surface_tri = calculate_TRI(domea[1, :, :].unsqueeze(0))
print("Mean bed 5x5 TRI:", np.round(torch.mean(domea_bed_tri).item(), 2))
print("Mean surface 5x5 TRI:", np.round(torch.mean(domea_surface_tri).item(), 2))

Km in y direction: 600.0
Km in x direction: 600.0
Correlation coefficients:


,bed,surface,thickness,mask,vy,vx,speed,angle
bed,1.00,0.53,-0.74,NaN,-0.62,-0.49,-0.61,-0.26
surface,0.53,1.00,0.18,NaN,-0.68,-0.77,-0.78,-0.01
thickness,-0.74,0.18,1.00,NaN,0.18,-0.05,0.09,0.30
mask,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vy,-0.62,-0.68,0.18,NaN,1.00,0.74,0.96,0.17
vx,-0.49,-0.77,-0.05,NaN,0.74,1.00,0.88,0.06
speed,-0.61,-0.78,0.09,NaN,0.96,0.88,1.00,0.11
angle,-0.26,-0.01,0.30,NaN,0.17,0.06,0.11,1.00


Mean bed 5x5 TRI: 169.06
Mean surface 5x5 TRI: 14.67


In [28]:
domea = torch.cat((domea, domea_bed_tri, domea_surface_tri), dim = 0)

torch.Size([12, 1201, 1201])

In [36]:
2**32


4294967296

In [33]:
torch.save(domea, '/home/kim/data/domea_tensor.pt')